# 事件式冻结三状态五列导出

本 Notebook 从唯一原始现货文件重算八状态和基础三状态，并分别重建下侧、上侧完整候选池，在 Development+Validation 中独立冻结。

最终导出五列：`date`、`three_state`、`minus_entry_signal`、`plus_entry_signal`、`final_three_state`。正式逻辑只在实际发出信号的当天把基础 0 改为 -1/1；没有新信号的后续 0 日不延续重标。同日双侧冲突保持 0。

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import Markdown, display

PACKAGE_ROOT = next(
    candidate for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'src' / 'pool_registry.py').is_file()
)
sys.path.insert(0, str(PACKAGE_ROOT / 'src'))
from event_signal_export import SIGNAL_COLUMNS, build_frozen_event_signal_export
from runtime_paths import resolve_output_dir, resolve_spot_path

SPOT_PATH = resolve_spot_path()
OUTPUT_DIR = resolve_output_dir(PACKAGE_ROOT)
print('输入现货：', SPOT_PATH)
print('输出目录：', OUTPUT_DIR)

## 运行：两侧独立冻结并生成五列信号

本格会打印现货读取、候选扫描、Dev/Val 筛选、冻结和冻结后 Test 的进度。两侧分别运行，不共享候选排序；Test 只在冻结之后计算。

In [ ]:
print('开始构造事件式五列三状态；两侧各自重建完整候选池。', flush=True)
signal, metadata, freezes = build_frozen_event_signal_export(
    SPOT_PATH, OUTPUT_DIR, show_progress=True
)
assert metadata['test_used_for_selection'] is False
assert list(signal.columns) == list(SIGNAL_COLUMNS)
print('事件式三状态导出完成。', flush=True)

## 冻结摘要和输入审计

In [ ]:
freeze_summary = pd.DataFrame(
    [
        {
            '侧别': side,
            '冻结候选': row['candidate_id'],
            '来源版本': row['source_version'],
            '核心逻辑': row['core_logic_name'],
            'Test是否用于筛选': row['test_used_for_selection'],
        }
        for side, row in metadata['freeze'].items()
    ]
)
display(freeze_summary)
display(pd.DataFrame({
    '项目': ['输入文件', '基础状态行数', '基础状态计数', '事件后状态计数', '下侧信号日', '上侧信号日', '同日冲突日', 'Test是否用于筛选'],
    '结果': [
        metadata['input_file'],
        metadata['spot_audit']['state_panel_rows'],
        metadata['state_counts_base'],
        metadata['state_counts_final'],
        metadata['signal_counts']['minus_entry_days'],
        metadata['signal_counts']['plus_entry_days'],
        metadata['same_day_base_zero_conflict_count'],
        metadata['test_used_for_selection'],
    ]
}))

## 最终五列输出（适合截图）

`final_three_state` 是事件式结果：只有当天 `minus_entry_signal=1` 或 `plus_entry_signal=1` 且基础状态为 0 时才改变；后续日期不自动延续。完整 CSV 已同时保存到输出目录。

In [ ]:
display(Markdown('### 最近 20 行'))
display(signal.tail(20))
display(Markdown('### 五列字段与状态计数'))
display(pd.DataFrame({'字段': list(signal.columns)}))
display(pd.DataFrame({
    '状态': [-1, 0, 1],
    '基础三状态天数': [int((signal['three_state'] == value).sum()) for value in [-1, 0, 1]],
    '事件后三状态天数': [int((signal['final_three_state'] == value).sum()) for value in [-1, 0, 1]],
}))
print('完整五列 CSV：', OUTPUT_DIR / metadata['csv_file'])
print('导出审计 JSON：', OUTPUT_DIR / metadata['json_file'])